# MISO/Q1 aggregate (Kaggle CPU là đủ)

Gộp toàn bộ shard zip thành bảng/hình cuối cùng cho Chương 4.

**Chuẩn bị:** upload TẤT CẢ file `miso_q1__*.zip` (40 file, mỗi cặp algo-seed
đúng 1 file — nếu một shard chạy lại nhiều lần chỉ giữ bản thành công mới nhất)
làm một Kaggle Dataset tên `miso-q1-shards`, và Add Input cùng dataset
`star-ris-miso-src`. Internet = On.

In [ ]:
SRC_ZIP = "/kaggle/input/star-ris-miso-src/star_ris_miso_src_a8f636f.zip"
SHARDS_DIR = "/kaggle/input/miso-q1-shards"
FROZEN_SOURCE_SHA = "2cb81296b206e92483347ede709ad4e53e5fa72182e05842d57083b7a0ac668c"
FINAL_PAPER = True   # False + --allow-partial chỉ dành cho pilot thiếu shard

In [ ]:
import os, sys, glob, json, shutil, subprocess, zipfile
WORK = "/kaggle/tmp/agg"; shutil.rmtree(WORK, ignore_errors=True)
SRC = os.path.join(WORK, "src"); SH = os.path.join(WORK, "shards")
os.makedirs(SRC); os.makedirs(SH)
with zipfile.ZipFile(SRC_ZIP) as z: z.extractall(SRC)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium", "pyyaml", "tqdm"], check=True)
sys.path.insert(0, SRC)
from main import source_tree_sha256
assert source_tree_sha256(SRC) == FROZEN_SOURCE_SHA
zips = sorted(glob.glob(os.path.join(SHARDS_DIR, "*.zip")))
print(f"{len(zips)} shard zips")
for zp in zips:
    with zipfile.ZipFile(zp) as z: z.extractall(SH)
flag = ["--final-paper"] if FINAL_PAPER else ["--allow-partial"]
r = subprocess.run([sys.executable, "main.py", "--config", "config/config.yaml",
                    "--aggregate-only", "--input-run-dirs", SH, "--out", WORK] + flag,
                   cwd=SRC)
assert r.returncode == 0, "AGGREGATE FAILED - đọc log phía trên"

In [ ]:
# ==== ĐÓNG GÓI KẾT QUẢ CUỐI (nhỏ, đủ cho Chương 4) ====
run_dirs = sorted(glob.glob(os.path.join(WORK, "results_revised", "aggregate_*")))
run = run_dirs[-1]
keep = ["tables", "figures", "results_summary.md", "run_meta.json",
        "completed_runs.csv", "effective_config.yaml",
        "scenario_bank_validation.json", "scenario_bank_test.json"]  # .json = hash/provenance, bỏ .npz nặng
final = "/kaggle/working/miso_q1_final_results.zip"
with zipfile.ZipFile(final, "w", zipfile.ZIP_DEFLATED) as z:
    for k in keep:
        p = os.path.join(run, k)
        if os.path.isdir(p):
            for root, _, files in os.walk(p):
                for f in files:
                    fp = os.path.join(root, f)
                    z.write(fp, os.path.relpath(fp, run))
        elif os.path.exists(p):
            z.write(p, k)
print(final, f"{os.path.getsize(final)/1e6:.1f} MB")
print(open(os.path.join(run, "results_summary.md")).read()[:2000])